In [1]:
%pip install "pandas==2.3.3" "numpy==1.26.4" "scipy==1.11.4" "joblib==1.3.2" "scikit-learn==1.7.2" "tpot==1.1.0" "setuptools<81"


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score
from sklearn.base import clone
from tpot import TPOTClassifier
import time
import os
import psutil

/opt/homebrew/lib/python3.10/site-packages/stopit/__init__.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
df = pd.read_csv("../../data/creditcard.csv")

In [4]:
# =========================
# COLUNAS
# =========================

X = df.drop(['Class'], axis=1)
y = df['Class'].astype(int)


In [ ]:
# =========================
# FUNÇÃO AUXILIAR DE RECURSOS
# =========================
def get_memory_usage_mb():
    """Retorna o uso atual de memória RAM do processo em Megabytes."""
    process = psutil.Process(os.getpid())
    return round(process.memory_info().rss / (1024 * 1024), 2)

# =========================
# INSTALAR TPOT 
# =========================
# %pip install tpot psutil

for SEED in [42, 101, 223, 302, 404]:
    print(f"\n================================")
    print(f"EXECUTANDO COM SEED: {SEED}")
    print(f"================================")

    # =========================
    # DIVISÃO TREINO/TESTE
    # =========================
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )

    classifier_config = {
        'sklearn.ensemble.RandomForestClassifier': {
            'n_estimators': [100],
            'criterion': ['gini', 'entropy'],
            'max_depth': [10, 20, None],
            'class_weight': ['balanced']
        },
        'sklearn.tree.DecisionTreeClassifier': {
            'criterion': ['gini', 'entropy'],
            'max_depth': [5, 10, None],
            'class_weight': ['balanced']
        },
        'xgboost.XGBClassifier': {
            'n_estimators': [100],
            'max_depth': [3, 5, 7],
            'scale_pos_weight': [10]
        }
    }

    tpot = TPOTClassifier(
        scorers="f1",
        cv=5,
        max_time_mins=60,
        max_eval_time_mins=10,
        search_space=classifier_config,
        random_state=SEED,
        n_jobs=1,
    )

    # =========================
    # TREINAMENTO DO TPOT + MÉTRICA COMPUTACIONAL
    # =========================
    print("Iniciando otimização com TPOT...")
    mem_antes_treino = get_memory_usage_mb()
    inicio_treino = time.perf_counter()

    tpot.fit(X_train, y_train)

    fim_treino = time.perf_counter()
    tempo_treino_total = round(fim_treino - inicio_treino, 2)
    mem_depois_treino = get_memory_usage_mb()

    pipelines_avaliados = tpot.evaluated_individuals
    coluna_score = "f1_score"

    if coluna_score not in pipelines_avaliados.columns:
        raise KeyError(
            f"A coluna {coluna_score!r} não foi encontrada. "
            f"Colunas disponíveis: {list(pipelines_avaliados.columns)}"
        )

    melhores_pipelines = (
        pipelines_avaliados
        .dropna(subset=[coluna_score, "Instance"])
        .sort_values(coluna_score, ascending=False)
        .head(5)
    )

    if melhores_pipelines.empty:
        raise RuntimeError("O TPOT não encontrou pipelines avaliados válidos.")

    print("\n===== 5 MELHORES PIPELINES (VALIDAÇÃO CRUZADA) SEED {} =====".format(SEED))
    print(melhores_pipelines[[coluna_score]].to_string())

    # =========================
    # MÉTRICAS DOS 5 MELHORES NO TESTE
    # =========================
    resultados = []
    y_true = y_test.to_numpy()

    for rank, (indice, linha) in enumerate(melhores_pipelines.iterrows(), start=1):
        modelo = clone(linha["Instance"])
        
        # Tempo de inferência para cada pipeline avaliado
        inicio_inferencia = time.perf_counter()
        
        y_pred = modelo.predict(X_test)

        if hasattr(modelo, "predict_proba"):
            probabilidades = modelo.predict_proba(X_test)
            classes = list(modelo.classes_)
            indice_positivo = classes.index(1)
            y_score = probabilidades[:, indice_positivo]
        elif hasattr(modelo, "decision_function"):
            y_score = modelo.decision_function(X_test)
            if list(modelo.classes_).index(1) == 0:
                y_score = -y_score
        else:
            y_score = y_pred
            
        tempo_inferencia = time.perf_counter() - inicio_inferencia

        resultados.append({
            "Posição": rank,
            "Índice TPOT": indice,
            "Pipeline": str(modelo),
            "Average Precision CV": round(float(linha[coluna_score]), 4),
            "Accuracy": round(accuracy_score(y_true, y_pred), 4),
            "Precision": round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
            "Recall": round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
            "F1-score Weighted": round(f1_score(y_true, y_pred, average="weighted", zero_division=0), 4),
            "F1-score": round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
            "Average Precision (teste)": round(average_precision_score(y_true, y_score), 4),
            "Seed": SEED,
            "Tempo Inferência (s)": round(tempo_inferencia, 4),
            "Tempo Treino Total (s)": tempo_treino_total,
            "Uso RAM Treino (MB)": mem_depois_treino
        })

    df_resultados = pd.DataFrame(resultados)
    print("\n===== MÉTRICAS DOS 5 MELHORES PIPELINES (TESTE) =====")
    print(df_resultados.to_string(index=False))

    print("\n===== MELHOR MODELO SELECIONADO PELO TPOT =====")
    print(tpot.fitted_pipeline_)


EXECUTANDO COM SEED: 42
Iniciando otimização com TPOT...


TypeError: TPOTEstimator.__init__() got an unexpected keyword argument 'config_dict'